In [27]:
import sys
import asyncio
import time
import os

import numpy as np

from lsst.ts import salobj

from lsst.ts.observatory.control.auxtel.atcs import ATCS
from lsst.ts.observatory.control.auxtel.latiss import LATISS
from lsst.ts.observatory.control.utils import RotType

In [2]:
# for tab completion to work in current notebook instance
%config IPCompleter.use_jedi = False

In [3]:
import logging
stream_handler = logging.StreamHandler(sys.stdout)
logger = logging.getLogger()
logger.addHandler(stream_handler)
logger.level = logging.DEBUG
# Make matplotlib less chatty
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [4]:
#Start classes
domain = salobj.Domain()
await asyncio.sleep(10) # This can be removed in the future...
atcs = ATCS(domain)
latiss = LATISS(domain)
await asyncio.gather(atcs.start_task, latiss.start_task)

atmcs: Adding all resources.
atptg: Adding all resources.
ataos: Adding all resources.
atpneumatics: Adding all resources.
athexapod: Adding all resources.
atdome: Adding all resources.
atdometrajectory: Adding all resources.
atcamera: Adding all resources.
atspectrograph: Adding all resources.
atheaderservice: Adding all resources.
atarchiver: Adding all resources.
Read historical data in 0.01 sec
Read 100 history items for RemoteEvent(ATDome, 0, allAxesInPosition)
Read 38 history items for RemoteEvent(ATDome, 0, appliedSettingsMatchStart)
Read 1 history items for RemoteEvent(ATDome, 0, authList)
Read 100 history items for RemoteEvent(ATDome, 0, azimuthCommandedState)
Read 100 history items for RemoteEvent(ATDome, 0, azimuthInPosition)
Read 100 history items for RemoteEvent(ATDome, 0, azimuthState)
Read 100 history items for RemoteEvent(ATDome, 0, doorEncoderExtremes)
Read 30 history items for RemoteEvent(ATDome, 0, dropoutDoorCommandedState)
Read 11 history items for RemoteEvent(ATDo

[[None, None, None, None, None, None, None], [None, None, None, None]]

trajectory DDS read queue is filling: 37 of 100 elements
torqueDemand DDS read queue is filling: 37 of 100 elements
nasymth_m3_mountMotorEncoders DDS read queue is filling: 37 of 100 elements
mount_Nasmyth_Encoders DDS read queue is filling: 37 of 100 elements
mount_AzEl_Encoders DDS read queue is filling: 38 of 100 elements
mount_AzEl_Encoders python read queue is filling: 37 of 100 elements
measuredTorque DDS read queue is filling: 38 of 100 elements
measuredMotorVelocity DDS read queue is filling: 38 of 100 elements
azEl_mountMotorEncoders DDS read queue is filling: 38 of 100 elements


In [ ]:
# enable components
#await atcs.enable()
#await latiss.enable()

In [19]:
# await atcs.prepare_for_flatfield()
# Because of the rotator problems, need to do this manually
tel_flat_el = 39.0
tel_flat_az = 205.7
dome_flat_az = 20.0

await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.DISABLED)
await atcs.open_m1_cover()
atcs.check.atdometrajectory = False

await atcs.point_azel(
target_name="FlatField position",
az=tel_flat_az,
el=tel_flat_el,
rot_tel=-40.0,
wait_dome=False,
)

await atcs.stop_tracking()
atcs.check.atdome = True
await atcs.slew_dome_to(dome_flat_az)


Cover state <MirrorCoverState.OPENED: 7>
M1 cover already opened.
Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = +000.000 deg 
Axes in position.
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the telescope.
process as completed...
atdometrajectory: <State.DISABLED: 1>
[Dome] delta Az = +003.060 deg
ATDome in position.
Axes in position.


In [90]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021030900142])

In [21]:
# Take 50 biases seq # 2-51
# Added wait to stop killing the recent images
for i in range(50):
    await asyncio.sleep(2.0)
    await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
logMessage DDS read queue is filling: 13 of 100 elements
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_id
BIAS 0001 - 0001
Generating group_i

In [22]:
# Take 10 10 second darks 52-61
await latiss.take_darks(10.0, 10)

Generating group_id
DARK 0001 - 0010
DARK 0002 - 0010
DARK 0003 - 0010
DARK 0004 - 0010
DARK 0005 - 0010
DARK 0006 - 0010
DARK 0007 - 0010
DARK 0008 - 0010
DARK 0009 - 0010
DARK 0010 - 0010


array([2021030900052, 2021030900053, 2021030900054, 2021030900055,
       2021030900056, 2021030900057, 2021030900058, 2021030900059,
       2021030900060, 2021030900061])

In [23]:
# Take 10 2 second flats 62-71
await latiss.take_flats(2.0, 10, filter='RG610', grating='empty_1')

Generating group_id
FLAT 0001 - 0010
FLAT 0002 - 0010
FLAT 0003 - 0010
FLAT 0004 - 0010
FLAT 0005 - 0010
logMessage DDS read queue is filling: 63 of 100 elements
FLAT 0006 - 0010
FLAT 0007 - 0010
FLAT 0008 - 0010
FLAT 0009 - 0010
FLAT 0010 - 0010


array([2021030900062, 2021030900063, 2021030900064, 2021030900065,
       2021030900066, 2021030900067, 2021030900068, 2021030900069,
       2021030900070, 2021030900071])

In [24]:
# Take flats for PTC 72-111
# Added wait to stop killing the recent images
for i in range(20):
    exp = 0.2 * float(i+1)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')
    if exp < 2.0:
        await asyncio.sleep(2.0)
    await latiss.take_flats(exp, 1, filter='empty_1', grating='empty_1')


Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
Generating group_id
FLAT 0001 - 0001
G

In [25]:
await atcs.stop_tracking()

Stop tracking.


In [26]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
#await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)

In [36]:
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [33]:
current_az = 90.0
current_el = 75.0
current_rot = -40.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +034.996 deg; delta Az= -109.996 deg; delta N1 = -000.000 deg; delta N2 = +019.964 deg 
[Telescope] delta Alt = +032.777 deg; delta Az= -107.652 deg; delta N1 = -000.000 deg; delta N2 = +019.284 deg 
[Telescope] delta Alt = +026.816 deg; delta Az= -101.656 deg; delta N1 = -000.000 deg; delta N2 = +015.981 deg 
[Telescope] delta Alt = +020.967 deg; delta Az= -095.657 deg; delta N1 = -000.000 deg; delta N2 = +010.489 deg 
[Telescope] delta Alt = +015.403 deg; delta Az= -089.656 deg; delta N1 = -000.000 deg; delta N2 = +004.801 deg 
[Telescope] delta Alt = +010.378 deg; delta Az= -083.656 deg; delta N1 = -000.000 deg; delta N2 = +001.058 deg 
[Telescope] delta Alt = +006.179 deg; delta Az= -077.657 deg; delta N1 = -000.000 deg; delta N2 = +000.1

In [34]:
await atcs.close_m1_cover()

Cover state <MirrorCoverState.OPENED: 7>
Closing M1 cover.
Cover state <MirrorCoverState.INMOTION: 8>
Cover state <MirrorCoverState.CLOSED: 6>


In [37]:
# Need a similar script for prepare for on sky

await atcs.home_dome()
await atcs.slew_dome_to(az=90.0)
try:
    scb = await atcs.rem.atdome.evt_scbLink.aget(timeout=atcs.fast_timeout)
except asyncio.TimeoutError:
    print(
        "Timed out waiting for ATDome CSC scbLink status event. Can not "
        "determine if CSC has communication with Shutter Control Box."
        "If running this on a jupyter notebook you may try to add an"
        "await asyncio.sleep(1.) before calling startup again to give the"
        "remotes time to get information from DDS. You may also try to "
        "re-cycle the ATDome CSC state to STANDBY and back to ENABLE."
        "Cannot continue."
    )
    sys.exit()

if not scb.active:
    print(
        "Dome CSC has no communication with Shutter Control Box. "
        "Dome controllers may need to be rebooted for connection to "
        "be established. Cannot continue."
    )
    sys.exit()


await atcs.open_dome_shutter()

await atcs.open_m1_cover()
await atcs.open_m1_vent()

await atcs.enable(settings=settings)

await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
)

Dome azimuth still homing.
Dome azimuth still homing.
Sending ATDomeTrajectory to DISABED state. Component will be left in DISABLEDstate or else it may send the ATDome back to alignment with the telescope.
process as completed...
atdometrajectory: <State.DISABLED: 1>
[Dome] delta Az = +090.000 deg
[Dome] delta Az = +089.900 deg
[Dome] delta Az = +089.550 deg
[Dome] delta Az = +088.970 deg
[Dome] delta Az = +088.160 deg
[Dome] delta Az = +087.110 deg
[Dome] delta Az = +085.830 deg
[Dome] delta Az = +084.300 deg
[Dome] delta Az = +082.560 deg
[Dome] delta Az = +080.570 deg
[Dome] delta Az = +078.370 deg
[Dome] delta Az = +075.910 deg
[Dome] delta Az = +073.250 deg
[Dome] delta Az = +070.350 deg
[Dome] delta Az = +067.230 deg
[Dome] delta Az = +063.880 deg
[Dome] delta Az = +060.410 deg
[Dome] delta Az = +056.930 deg
[Dome] delta Az = +053.470 deg
[Dome] delta Az = +050.000 deg
[Dome] delta Az = +046.570 deg
[Dome] delta Az = +043.140 deg
[Dome] delta Az = +039.690 deg
[Dome] delta Az = +

AckError: msg='Command failed', ackcmd=(ackcmd private_seqNum=1572119168, ack=<SalRetCode.CMD_NOPERM: -300>, error=0, result='ERROR: Command openCover rejected while in EnabledState state.')

In [39]:
await salobj.set_summary_state(atcs.rem.ataos, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [40]:
await atcs.open_m1_cover()

Cover state <MirrorCoverState.CLOSED: 6>
Opening M1 cover.
Cover state <MirrorCoverState.INMOTION: 8>
Cover state <MirrorCoverState.OPENED: 7>


In [41]:
await atcs.open_m1_vent()

M1 vent state <VentsPosition.CLOSED: 1>
Opening M1 vents.
M1 vent state <VentsPosition.PARTIALLYOPENED: 2>
M1 vent state <VentsPosition.OPENED: 0>


In [43]:
await atcs.enable()

Enabling all components
Gathering settings.
Couldn't get settingVersions event. Using empty settings.
Complete settings for atmcs.
Complete settings for atptg.
Complete settings for ataos.
Complete settings for atpneumatics.
Complete settings for athexapod.
Complete settings for atdome.
Complete settings for atdometrajectory.
Settings versions: {'atmcs': '                                                                                                                               ', 'atptg': '', 'ataos': 'current', 'atpneumatics': '                                                                                                                               ', 'athexapod': 'summit', 'atdome': 'test', 'atdometrajectory': ''}
[atmcs]::[<State.ENABLED: 2>]
[atptg]::[<State.ENABLED: 2>]
[ataos]::[<State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]
[atpneumatics]::[<State.ENABLED: 2>]
[athexapod]::[<State.ENABLED: 2>]
[atdome]::[<State.ENABLED: 2>]
[atdometrajectory]::[<State.DISABL

In [44]:
await atcs.rem.ataos.cmd_enableCorrection.set_start(
    m1=True, hexapod=True, atspectrograph=True, timeout=atcs.long_timeout
)

In [ ]:
# HD55070 m = 5.45

In [86]:
current_az = 90.0
current_el = 75.0
current_rot = -40.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = +009.616 deg; delta Az= +003.307 deg; delta N1 = +000.000 deg; delta N2 = +007.749 deg 
[Telescope] delta Alt = +006.346 deg; delta Az= +000.276 deg; delta N1 = -000.000 deg; delta N2 = +002.673 deg 
[Telescope] delta Alt = +001.046 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.001 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = +000.004 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -000.002 deg 
Telescope in position.


In [29]:
atcs.slew_object?

Signature:
atcs.slew_object(
    name,
    rot=0.0,
    rot_type=<RotType.SkyAuto: 1>,
    dra=0.0,
    ddec=0.0,
    slew_timeout=240.0,
)
Docstring:
Slew to an object name.

Use simbad to resolve the name and get coordinates.

Parameters
----------
name : `str`
    Target name.
rot : `float`, `str` or `astropy.coordinates.Angle`
    Specify desired rotation angle. Strategy depends on `rot_type`
    parameter. Accepts float (deg), sexagesimal string (DD:MM:SS.S or
    DD MM SS.S) coordinates or `astropy.coordinates.Angle`
rot_type :  `lsst.ts.observatory.control.utils.RotType`
    Rotation type. This parameter defines how `rot_value` is threated.
    Default is `SkyAuto`, the rotator is positioned with respect to the
    North axis and is automacally wrapped if outside the limit. See
    `RotType` for more options.
dra : `float`
    Differential Track Rate in RA (arcsec/second). Default is 0.
ddec : `float`
    Differential Track Rate in Dec (arcsec/second). Default is 0.
slew_timeout

In [70]:
await salobj.set_summary_state(atcs.rem.atptg, salobj.State.ENABLED)

[<State.FAULT: 3>, <State.STANDBY: 5>, <State.DISABLED: 1>, <State.ENABLED: 2>]

In [82]:
# HD55070 m = 5.45, Az = +85, Alt = +74
# HD78876 m = 5.45, Az = +90, Alt = +60
await atcs.slew_object('HD78876', rot=50.0, rot_type=RotType.PhysicalSky)

Slewing to HD78876: 09 09 45.7965 -25 48 11.192
Setting rotator physical position to 50.0 deg. Rotator will track sky.
Parallactic angle: -106.54625438203846 | Sky Angle: -40.26091756925956
xml 7 compatibility mode: rotPA=-40.26091756925956, rotFrame=1
Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
atdometrajectory: <State.ENABLED: 2>
[Telescope] delta Alt = +000.014 deg; delta Az= -000.013 deg; delta N1 = -000.000 deg; delta N2 = -002.803 deg 
[Telescope] delta Alt = -000.002 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -000.295 deg 
Got True
Waiting for telescope to settle.
[Telescope] delta Alt = -000.002 deg; delta Az= +000.002 deg; delta N1 = +000.000 deg; delta N2 = -000.006 deg 
Telescope in position.


In [83]:
await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')

Generating group_id
OBJECT 0001 - 0001


array([2021030900141])

In [59]:
await atcs.rem.ataos.cmd_offset.set_start(z=-0.1)

In [68]:
await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')

Generating group_id
OBJECT 0001 - 0001


array([2021030900127])

In [74]:
z_offset = -0.20
print(z_offset)
await atcs.rem.ataos.cmd_offset.set_start(z=z_offset)
await asyncio.sleep(2)
for i in range(9):
    await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')
    z_offset += 0.05
    print(z_offset)
    await atcs.rem.ataos.cmd_offset.set_start(z=z_offset)
    await asyncio.sleep(2)
    

-0.2
Generating group_id
OBJECT 0001 - 0001
-0.15000000000000002
Generating group_id
OBJECT 0001 - 0001
-0.10000000000000002
Generating group_id
OBJECT 0001 - 0001
-0.05000000000000002
Generating group_id
OBJECT 0001 - 0001
-1.3877787807814457e-17
Generating group_id
OBJECT 0001 - 0001
0.04999999999999999
logMessage DDS read queue is filling: 20 of 100 elements
Generating group_id
OBJECT 0001 - 0001
0.09999999999999999
Generating group_id
OBJECT 0001 - 0001
0.15
Generating group_id
OBJECT 0001 - 0001
0.2
Generating group_id
OBJECT 0001 - 0001
0.25
logMessage DDS read queue is filling: 20 of 100 elements


In [75]:
focus_summary = await atcs.rem.ataos.evt_focusOffsetSummary.next(flush=False, timeout=5)

In [76]:
focus_summary.print_vars()

private_revCode: d8296953, private_sndStamp: 1615334415.2704034, private_rcvStamp: 1615334415.2708309, private_seqNum: 73, private_identity: ATAOS, private_origin: 149691, private_host: 0, total: 0.0, userApplied: 0.0, filter: 0.0, disperser: 0.0, wavelength: 0.0, priority: 0


In [80]:
await atcs.rem.ataos.cmd_offset.set_start(z=0.0)

In [89]:
# To reset all offsets
tmp = await atcs.rem.ataos.cmd_resetOffset.set_start(axis='z')

In [81]:
await latiss.take_object(1.0, 1, filter='empty_1', grating='empty_1')

Generating group_id
OBJECT 0001 - 0001


array([2021030900140])

In [87]:
await atcs.close_m1_cover()

Cover state <MirrorCoverState.OPENED: 7>
Closing M1 cover.
Cover state <MirrorCoverState.INMOTION: 8>
Cover state <MirrorCoverState.CLOSED: 6>


In [88]:
await atcs.close_dome()

Closing dome shutter...
process as completed...
atdome: <State.ENABLED: 2>
Waiting for ATDome mainDoorState: <ShutterDoorState.CLOSED: 1>. Current state: <ShutterDoorState.OPENED: 2>.
mainDoorState: <ShutterDoorState.CLOSING: 5>
mainDoorState: <ShutterDoorState.CLOSED: 1>
Finishing ATDome shutter command task.
ATDome shutter command task not done. Cancelling.
ATDome shutter command task cancelled.


In [91]:
# Looking at banding while the dome is closed
current_az = 90.0
current_el = 75.0
current_rot = -110.0
await atcs.point_azel(current_az, current_el, rot_tel=current_rot)

Sending command
Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.
Scheduling check coroutines
process as completed...
atmcs: <State.ENABLED: 2>
atptg: <State.ENABLED: 2>
ataos: <State.ENABLED: 2>
atpneumatics: <State.ENABLED: 2>
athexapod: <State.ENABLED: 2>
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = -000.000 deg; delta N2 = -069.712 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.001 deg; delta N1 = +000.000 deg; delta N2 = -064.390 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -058.388 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = +000.000 deg; delta N2 = -052.386 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= +000.000 deg; delta N1 = -000.000 deg; delta N2 = -046.377 deg 
[Telescope] delta Alt = -000.000 deg; delta Az= -000.000 deg; delta N1 = -000.000 deg; delta N2 = -040.407 deg 
[Telescope] delta Alt = -000.000

In [92]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021030900143])

In [93]:
await atcs.stop_tracking()

Stop tracking.
Unknown tracking state: 9
Unknown tracking state: 10
In Position: True.


In [94]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [95]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021030900144])

In [96]:
await latiss.take_bias(1)

Generating group_id
BIAS 0001 - 0001


array([2021030900145])

In [99]:
# Ferrite clamp on 146, 147, 148
await latiss.take_bias(1)
# Took clamp back off.`1

Generating group_id
BIAS 0001 - 0001


array([2021030900148])

In [100]:
await latiss.standby()

[atcamera]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atspectrograph]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atheaderservice]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atarchiver]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.


In [101]:
await atcs.close_m1_vent()

M1 vent state <VentsPosition.OPENED: 0>
Closing M1 vents.
M1 vent state <VentsPosition.CLOSED: 1>


In [102]:
await atcs.close_m1_cover()

Cover state <MirrorCoverState.CLOSED: 6>
M1 cover already closed.


In [103]:
await atcs.stop_tracking()

Stop tracking.
Unknown tracking state: 6


TimeoutError: 

In [104]:
# Putting everything back in standby.
await atcs.standby()

[atmcs]::[<State.STANDBY: 5>]
[atptg]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[ataos]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atpneumatics]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[athexapod]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
[atdome]::<State.ENABLED: 2>
[atdometrajectory]::[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]
All components in <State.STANDBY: 5>.


In [ ]:
await salobj.set_summary_state(atcs.rem.atmcs, salobj.State.STANDBY)

In [105]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.STANDBY, settingsToApply="test")

[<State.ENABLED: 2>, <State.DISABLED: 1>, <State.STANDBY: 5>]

In [ ]:
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [ ]:
# Disable the dome
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.DISABLED, settingsToApply="test")
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.DISABLED)

In [ ]:
# Take event checking out the slew commands to test telescope only
atcs.check.atdome = False
atcs.check.atdometrajectory = False

In [ ]:
await atcs.rem.atdome.cmd_start.set_start(settingsToApply="test", timeout=30)

In [ ]:
# Re-enable the dome
# take event checking out the slew commands to test telescope only
# otherwise it'll wait for the dome before completing slew command
atcs.check.atdome = True
atcs.check.atdometrajectory = True

In [ ]:
await salobj.set_summary_state(atcs.rem.atdome, salobj.State.ENABLED, settingsToApply="test")

In [ ]:
await salobj.set_summary_state(atcs.rem.atdometrajectory, salobj.State.ENABLED)